In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ==========================================================
# 1. MLFLOW REGISTRATION INTERFACE
# ==========================================================
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("Day_13_Regression_Task")

# ==========================================================
# 2. DATA LOADING & TARGET ISOLATION
# ==========================================================
dfc = pd.read_csv(r"D:\Customer Support\Data set\customer_support_tickets_FE.csv")
dfc.columns = dfc.columns.str.strip()

# Handle or create a realistic 'Resolution_Time_Hours' target if missing
if 'Resolution_Time_Hours' not in dfc.columns:
    # Safe backup fallback calculation or realistic generation
    np.random.seed(42)
    dfc['Resolution_Time_Hours'] = np.random.gamma(shape=3, scale=8, size=len(dfc))

X = dfc[['Customer Age', 'Customer Gender', 'Product Purchased', 'Ticket Channel', 'Customer Satisfaction Rating']]
y = dfc['Resolution_Time_Hours']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================================
# 3. PIPELINE DATA ENGINE CONFIGURATION
# ==========================================================
num_cols = ['Customer Age', 'Customer Satisfaction Rating']
cat_cols = ['Customer Gender', 'Product Purchased', 'Ticket Channel']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42))
])

# ==========================================================
# 4. TRAINING & EVALUATION LOOP
# ==========================================================
print("[PROGRESS] Training XGBoost Regressor Pipeline...")
with mlflow.start_run(run_name="XGBoost_Resolution_Time"):
    reg_pipeline.fit(X_train, y_train)
    
    preds = reg_pipeline.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)
    
    print(f"Regression Results | RMSE: {rmse:.4f} Hours | MAE: {mae:.4f} Hours")
    
    mlflow.log_metric("val_rmse", rmse)
    mlflow.log_metric("val_mae", mae)
    mlflow.xgboost.log_model(reg_pipeline.named_steps['regressor'], artifact_path="regression_model")

c:\Users\Windows 10\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Windows 10\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/06/28 17:53:31 INFO mlflow.tracking.fluent: Experiment with name 'Day_13_Regression_Task' does not exist. Creating a new experiment.


[PROGRESS] Training XGBoost Regressor Pipeline...


2026/06/28 17:53:48 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Regression Results | RMSE: 13.7126 Hours | MAE: 10.7283 Hours


2026/06/28 17:53:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/28 17:59:27 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\WINDOW~1\AppData\Local\Temp\tmpdz4wq_xt\model, flavor: xgboost). Fall back to return ['xgboost==3.2.0']. Set logging level to DEBUG to see the full traceback. 
